In [26]:
# Needed if running on Colab, comment out if in local environment!
!pip3 install open-spiel
!pip3 install torch

  Using cached open_spiel-1.6.9.tar.gz (5.4 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached ml_collections-1.1.0-py3-none-any.whl.metadata (22 kB)
Using cached absl_py-2.3.1-py3-none-any.whl (135 kB)
Using cached ml_collections-1.1.0-py3-none-any.whl (76 kB)
Failed to build open-spiel


  error: subprocess-exited-with-error
  
  exit code: 1
  
  [681 lines of output]
  running bdist_wheel
  running build
  running build_py
  creating build\lib.win-amd64-cpython-310\open_spiel
  copying open_spiel\__init__.py -> build\lib.win-amd64-cpython-310\open_spiel
  creating build\lib.win-amd64-cpython-310\open_spiel\integration_tests
  copying open_spiel\integration_tests\api_test.py -> build\lib.win-amd64-cpython-310\open_spiel\integration_tests
  copying open_spiel\integration_tests\playthrough_test.py -> build\lib.win-amd64-cpython-310\open_spiel\integration_tests
  copying open_spiel\integration_tests\__init__.py -> build\lib.win-amd64-cpython-310\open_spiel\integration_tests
  creating build\lib.win-amd64-cpython-310\open_spiel\python
  copying open_spiel\python\observation.py -> build\lib.win-amd64-cpython-310\open_spiel\python
  copying open_spiel\python\policy.py -> build\lib.win-amd64-cpython-310\open_spiel\python
  copying open_spiel\python\rl_agent.py -> build\lib.w

In [4]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from go_search_problem import GoProblem, GoState
from heuristic_go_problems import GoProblemLearnedHeuristic, GoProblemSimpleHeuristic
from agents import GreedyAgent, RandomAgent, AlphaBetaAgent
import matplotlib.pyplot as plt
from tqdm import tqdm
from game_runner import GameRunner
import pickle

torch.set_default_tensor_type(torch.FloatTensor)

pyspiel not found, using PyGo instead
pygame 2.6.1 (SDL 2.28.4, Python 3.10.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


C:\Users\marpl\Documents\work\cs410\cs410_env\lib\site-packages\torch\__init__.py:1275: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\tensor\python_tensor.cpp:436.)
  _C._set_default_tensor_type(t)


In [5]:
def load_dataset(path: str):
    with open(path, 'rb') as f:
        dataset = pickle.load(f)
    return dataset

# We've provided a dataset with pyspiel and without (i.e., pygo)
# dataset_5x5 = load_dataset('dataset_5x5.pkl')
dataset_5x5 = load_dataset('dataset_5x5_pygo.pkl')

In [6]:
def save_model(path: str, model):
    """
    Save model to a file
    Input:
        path: path to save model to
        model: Pytorch model to save
    """
    torch.save({
        'model_state_dict': model.state_dict(),
    }, path)

def load_model(path: str, model):
    """
    Load model from file

    Note: you still need to provide a model (with the same architecture as the saved model))

    Input:
        path: path to load model from
        model: Pytorch model to load
    Output:
        model: Pytorch model loaded from file
    """
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model_state_dict'])
    return model

# Task 1: Convert GameState to Features

In [31]:
def get_features(game_state: GoState):
    """
    Map a game state to a list of features.

    Some useful functions from game_state include:
        game_state.size: size of the board
        get_pieces_coordinates(player_index): get coordinates of all pieces of a player (0 or 1)
        get_pieces_array(player_index): get a 2D array of pieces of a player (0 or 1)
        
        get_board(): get a 2D array of the board with 4 channels (player 0, player 1, empty, and player to move). 4 channels means the array will be of size 4 x n x n
    
        Descriptions of these methods can be found in the GoState

    Input:
        game_state: GoState to encode into a fixed size list of features
    Output:
        features: list of features




        Featruees we want are the 50 board positions, whos turn it is, and how many liberties each team has
    """
    board = game_state.get_board()
    
    # board[0] is BLACK pieces, board[1] is WHITE pieces
    black_board = board[0]  
    white_board = board[1]

    features = []
    
    # Add board positions for white pieces (25 features for 5x5 board)
    features.extend(white_board.flatten().tolist())
    
    # Add board positions for black pieces (25 features for 5x5 board)
    features.extend(black_board.flatten().tolist())

    # Add player to move (1 feature)
    features.append(game_state.player_to_move())

    # Get piece counts using GoState methods
    white_pieces = game_state.get_pieces_coordinates(1)  # WHITE is player 1
    black_pieces = game_state.get_pieces_coordinates(0)  # BLACK is player 0
    
    white_count = len(white_pieces)
    black_count = len(black_pieces)

    # Add piece count comparison features (2 features)
    features.append(1 if white_count > black_count else 0)
    features.append(1 if black_count > white_count else 0)  
    
    return features
    


In [8]:
# Print information about first data point
data_point = dataset_5x5[0]
features = get_features(data_point[0])
action = data_point[1]
result = data_point[2]
print(data_point[0])
print("features", features)
print("Action #", action)
print("Game Result", result)

{'board': array([[[0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 1.],
        [0., 0., 1., 0., 0.]],

       [[0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.]],

       [[1., 1., 0., 1., 1.],
        [1., 1., 0., 1., 1.],
        [1., 1., 0., 0., 1.],
        [1., 1., 1., 1., 0.],
        [1., 0., 0., 1., 1.]],

       [[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]]]), 'size': 5, 'player_to_move': 1, 'is_terminal': False, 'legal_actions': [0, 1, 3, 4, 5, 6, 8, 9, 10, 11, 14, 15, 16, 17, 18, 20, 23, 24, 25], 'black_pieces': array([[0, 2],
       [2, 2],
       [3, 4],
       [4, 2]]), 'white_pieces': array([[1, 2],
       [2, 3],
       [4, 1]])}
features [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0,

# Task 2: Supervised Learning of a Value Network

In [9]:
class ValueNetwork(nn.Module):
    def __init__(self, input_size):
        super(ValueNetwork, self).__init__()

        # Value function outputs a single number (the estimated value of the state)
        output_size = 1

        # Multi-layer network with non-linear activations
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, output_size)
        
        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()

    def forward(self, x):
        """
        Run forward pass of network

        Input:
            x: input to network
        Output:
            output of network (value between -1 and 1)
        """
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.tanh(self.fc4(x))  # Tanh to get output in [-1, 1] range (like game results)
        return x

In [11]:
# This will not produce meaningful outputs until trained, but you can test for syntax errors
features_tensor = torch.Tensor(features)
value_net = ValueNetwork(len(features))
print("predicted Value", value_net(features_tensor))

predicted Value tensor([-0.1404], grad_fn=<TanhBackward0>)


In [12]:
def train_value_network(dataset, num_epochs, learning_rate):
    """
    Train a value network on the provided dataset.

    Input:
        dataset: list of (state, action, result) tuples
        num_epochs: number of epochs to train for
        learning_rate: learning rate for gradient descent
    Output:
        model: trained model
    """
    # Make sure dataset is shuffled for better performance
    random.shuffle(dataset)
    
    # Split into train/test sets (80/20 split)
    split_idx = int(0.8 * len(dataset))
    train_dataset = dataset[:split_idx]
    test_dataset = dataset[split_idx:]
    
    # Get feature size from first data point
    first_state = dataset[0][0]
    features = get_features(first_state)
    input_size = len(features)
    
    # Create model
    model = ValueNetwork(input_size)

    # Mean Squared Error loss for regression
    loss_function = nn.MSELoss()

    # You can use Adam, which is stochastic gradient descent with ADAptive Momentum
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    batch_size = 32

    for epoch in range(num_epochs):
        model.train()
        batch_loss = 0
        batch_counter = 0
        epoch_loss = 0
        num_batches = 0
        
        for data_point in train_dataset:
            state = data_point[0]
            features = get_features(state)
            features_tensor = torch.tensor(features, dtype=torch.float32)

            # The game result is our label (from BLACK's perspective: +1 or -1)
            result = data_point[2]
            label = torch.tensor([result], dtype=torch.float32)

            # Get model prediction of value
            prediction = model(features_tensor)

            # Compute Loss for data point
            loss = loss_function(prediction, label)
            batch_loss += loss
            batch_counter += 1
            
            if batch_counter % batch_size == 0:
                # Call backward to run backward pass and compute gradients
                batch_loss.backward()

                # Run gradient descent step with optimizer
                optimizer.step()

                # Reset gradient for next batch
                optimizer.zero_grad()
                
                epoch_loss += batch_loss.item()
                num_batches += 1
                batch_loss = 0
        
        # Handle remaining data points if batch doesn't divide evenly
        if batch_counter % batch_size != 0:
            batch_loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            epoch_loss += batch_loss.item()
            num_batches += 1
        
        # Evaluate on test set
        model.eval()
        test_loss = 0
        with torch.no_grad():
            for data_point in test_dataset:
                state = data_point[0]
                features = get_features(state)
                features_tensor = torch.tensor(features, dtype=torch.float32)
                result = data_point[2]
                label = torch.tensor([result], dtype=torch.float32)
                prediction = model(features_tensor)
                test_loss += loss_function(prediction, label).item()
        
        avg_train_loss = epoch_loss / num_batches if num_batches > 0 else 0
        avg_test_loss = test_loss / len(test_dataset) if len(test_dataset) > 0 else 0
        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Test Loss: {avg_test_loss:.4f}")

    return model

value_model = train_value_network(dataset_5x5, 10, 1e-4)
save_model("value_model.pt", value_model)

Epoch 1/10, Train Loss: 29.7154, Test Loss: 0.7296
Epoch 2/10, Train Loss: 22.2839, Test Loss: 0.6662
Epoch 3/10, Train Loss: 21.4378, Test Loss: 0.6597
Epoch 4/10, Train Loss: 21.1689, Test Loss: 0.6570
Epoch 5/10, Train Loss: 20.9700, Test Loss: 0.6548
Epoch 6/10, Train Loss: 20.7858, Test Loss: 0.6527
Epoch 7/10, Train Loss: 20.6024, Test Loss: 0.6505
Epoch 8/10, Train Loss: 20.4095, Test Loss: 0.6483
Epoch 9/10, Train Loss: 20.2053, Test Loss: 0.6461
Epoch 10/10, Train Loss: 19.9849, Test Loss: 0.6436


## Comparing Learned Value function against other Agents

In [32]:
class GoProblemLearnedHeuristic(GoProblem):
    def __init__(self, model=None, state=None):
        super().__init__(state=state)
        self.model = model

    def encoding(self, state):
        """
        Get encoding of state (convert state to features)
        Note, this may call get_features() from Task 1. 

        Input:
            state: GoState to encode into a fixed size list of features
        Output:
            features: list of features
        """
        # Use get_features() from Task 1
        features = get_features(state)
        return features

    def heuristic(self, state, player_index):
        """
        Return heuristic (value) of current state

        Input:
            state: GoState to encode into a fixed size list of features
            player_index: index of player to evaluate heuristic for
        Output:
            value: heuristic (value) of current state
        """
        # Get features and convert to tensor
        features = self.encoding(state)
        features_tensor = torch.tensor(features, dtype=torch.float32)
        
        # Get model prediction (value from BLACK's perspective, player_index 1)
        with torch.no_grad():
            value = self.model(features_tensor).item()
        
        # If evaluating for WHITE (player_index 0), negate the value
        if player_index == 0:
            value = -value

        # Note, your agent may perform better if you force it not to pass
        # (i.e., don't select action #25 on a 5x5 board unless necessary)
        return value

    def __str__(self) -> str:
        return "Learned Heuristic"


def create_value_agent_from_model(value_model=None):
    """
    Create agent object from saved model. This (or other methods like this) will be how your agents will be created in gradescope and in the final tournament.
    """

    model_path = "value_model.pt"
    # Feature size: 25 (white board) + 25 (black board) + 1 (player to move) + 2 (piece count comparison) = 53
    feature_size = 53
    
    if value_model is None:
        model = ValueNetwork(feature_size)
        model = load_model(model_path, model)
    else:
        model = value_model
        
    heuristic_search_problem = GoProblemLearnedHeuristic(model)

    # Using GreedyAgent with the learned heuristic
    learned_agent = GreedyAgent(heuristic_search_problem)

    return learned_agent

learned_agent = create_value_agent_from_model(value_model)
agent2 = GreedyAgent(GoProblemSimpleHeuristic())
print("Greedy Agent", agent2)
print("Learned Agent", learned_agent)

game_runner = GameRunner()
game_runner.play_tournament(learned_agent, agent2, num_games=100)


learned_agent = create_value_agent_from_model(value_net)
agent2 = GreedyAgent(GoProblemSimpleHeuristic)
print("Greedy Agent", agent2)
print("Learned Agent", learned_agent)

game_runner = GameRunner()
game_runner.play_tournament(learned_agent, agent2, num_games=100)

Greedy Agent GreedyAgent + Simple Heuristic
Learned Agent GreedyAgent + Learned Heuristic


Playing tournament: 100%|█████████████████| 50/50 [01:04<00:00,  1.29s/it]


Tournament Results
Games played: 100
GreedyAgent + Learned Heuristic wins: 42 (42.0%)
GreedyAgent + Simple Heuristic wins: 58 (58.0%)
GreedyAgent + Learned Heuristic wins as BLACK: 0
GreedyAgent + Simple Heuristic wins as BLACK: 8
GreedyAgent + Learned Heuristic avg move time: 0.007s
GreedyAgent + Simple Heuristic avg move time: 0.005s
GreedyAgent + Learned Heuristic min time remaining: 26.9s
GreedyAgent + Simple Heuristic min time remaining: 26.9s
Greedy Agent GreedyAgent + <class 'heuristic_go_problems.GoProblemSimpleHeuristic'>
Learned Agent GreedyAgent + Learned Heuristic


Playing tournament:   0%|                          | 0/50 [00:00<?, ?it/s]


TypeError: GoProblem.get_available_actions() missing 1 required positional argument: 'state'